这两篇论文（《RL's Razor》与《Sparse but Critical》）虽然都聚焦于大模型强化学习（RL）微调中的“稀疏性（Sparsity）”，但它们观察的维度、得出的结论以及背后的物理意义截然不同。

简单来说：
- 《RL's Razor》探讨的是“<font color='red'>权重更新空间</font>”的假象，而
- 《Sparse but Critical》探讨的是“<font color='red'>生成序列空间</font>”的真相。

以下是这两篇 ICLR 2026 论文在 RL 微调领域的横向对比与核心观点梳理：
## 观察维度的差异
- 《RL's Razor》：<font color='red'>参数/权重级别（Weight-Level）</font>
  - 视角：关注模型内部参数在训练过程中的更新幅度。
  - 核心问题：RL 的权重更新为什么看起来那么稀疏？是算法本身保守，还是别的原因？
- 《Sparse but Critical》：<font color='red'>Token/生成级别（Token-Level）</font>
  - 视角：<font color='red'>关注模型输出序列在训练前后的概率分布变化。</font>
  - 核心问题：RL 微调到底改变了模型生成的哪些词？这些改变对最终性能有多大影响？
## 对“稀疏性”本质的解释（核心冲突点）
- 《RL's Razor》：稀疏性是“假象”（Illusion）
  - 观点：RL 的更新在本质上是稠密且满秩（Full Rank）的。你看到的稀疏，纯粹是因为使用了 bfloat16 低精度训练，导致大量微小的梯度被截断为 0。
  - 潜台词：<font color='red'>RL 其实对模型进行了全方位的微调，只是大部分更新幅度太小，被硬件精度“吃掉”了。</font>
- 《Sparse but Critical》：稀疏性是“真相”（Reality）
  - 观点：RL 带来的分布变化在逻辑上是极度稀疏的。
    - 在生成序列中，超过 80%-98% 的 Token 分布几乎没有变化，只有极少数关键决策点的 Token 发生了概率质量的重新分配。
  - 潜台词：RL 就像一把“手术刀”，只精准切中了推理链条中决定成败的几个关键节点，而不是盲目地重写所有输出。
  
## 性能提升的归因机制
- 《RL's Razor》：主要矛盾决定论
  - 模型性能的提升依赖于那些幅度较大的“主梯度”。<font color='red'>那些被 bfloat16 抹除的微小更新（即使它们让矩阵变成了满秩）其实对最终性能贡献极低，甚至是冗余的噪声。</font>
- 《Sparse but Critical》：蝴蝶效应与轨迹引导
  - 模型性能的提升完全由那一小撮“高散度（High Divergence）”的关键 Token 决定。<font color='red'>RL 并不“发明”新词，而是通过重新排序基础模型已有的候选词，像引导员一样将生成轨迹“推”向正确的推理路径。哪怕只替换 1% 的关键 Token，也能恢复全部性能。</font>
  
## 对实际训练的指导意义
- 《RL's Razor》的工程启示：精度与正则化
  - 在做 RL 研究和诊断时，必须使用 float32 以避免被精度误导。
  - 但在实际工程中，bfloat16 造成的“伪稀疏”其实起到了隐式正则化的作用，防止了模型过拟合或策略崩溃，因此它依然是安全且高效的。
- 《Sparse but Critical》的算法启示：资源优化
  - <font color='red'>既然 RL 只需要改变极少数的 Token，那么在训练时就可以把算力集中在刀刃上</font>。
    - 论文提出的<font color='blue'>“散度加权优势函数”就是给这些关键 Token 更高的学习权重，从而在提升性能的同时节省计算资源</font>。
  
## 💡 总结：如何统一这两种观点？
这两篇论文并不矛盾，它们从不同维度拼凑出了 RL 微调大模型的真实物理图景：
- 在微观参数层面：
  - RL 确实进行了极其广泛、细微的调整（《RL's Razor》证实了它是满秩的），这些调整在低精度下不可见。
- 在宏观生成层面：
  - 这些海量的微小参数调整，最终只“撬动”了生成序列中极少数的关键决策点（《Sparse but Critical》证实了输出的极度稀疏）。

### 一句话概括：
- RL 微调大模型，本质上是一场“牵一发而动全身”的精密手术。它在参数空间里进行了极其广泛且微小的试探（《RL's Razor》），但最终只在生成空间里精准地改变了那几个决定推理轨迹走向的关键路口（《Sparse but Critical》）。